In [ ]:
# hide
import numpy as np
import pyquist as pq
from scipy.signal import lfilter

In [ ]:
# hide
def pluck(pitch: float, duration: float, f_s: int = 44100, **kwargs) -> pq.Audio:
    """A Karplus-Strong plucked string (Chapter 12) that decays by 60 dB in `duration`
    seconds. An all-pass filter (Chapter 13) in the loop tunes it precisely."""
    P = f_s / pq.helper.pitch_to_frequency(pitch)   # the period, in (fractional) samples
    M = int(P - 0.6)                                 # the delay line (the low-pass adds 1/2 sample)
    D = P - 0.5 - M                                  # the all-pass adds the remaining D samples
    c = (1 - D) / (1 + D)                            # the all-pass gain for a delay of D samples
    g = 10 ** (-3 * P / (f_s * duration))            # decay by 60 dB over the duration
    x = np.zeros(int(duration * f_s))
    x[:M] = np.random.uniform(-1.0, 1.0, M)          # excitation: a burst of white noise
    # The whole loop is one recursive filter (Chapter 9), which SciPy runs quickly:
    # y[n] = x[n] + c x[n-1] - c y[n-1] + (g/2) (c y[n-M] + (1 + c) y[n-M-1] + y[n-M-2])
    a = np.zeros(M + 3)
    a[:2] = [1.0, c]
    a[M:] -= 0.5 * g * np.array([c, 1 + c, 1.0])
    return pq.Audio(lfilter([1.0, c], a, x), f_s)

In [ ]:
def compute_ngram(midi_path: str, n: int) -> tuple[list[int], np.ndarray]:
    """The states (pitches) of the melody of a MIDI file, and its n-gram probabilities:
    probabilities[a_1, ..., a_{n-1}, b] is the probability that states[b] follows
    the context states[a_1], ..., states[a_{n-1}]."""
    score, _ = pq.Score.from_midi(midi_path)
    highest = {}                                 # the melody: the highest pitch at each onset
    for time, kwargs in score:
        highest[time] = max(highest.get(time, 0), kwargs["pitch"])
    melody = [highest[time] for time in sorted(highest)]
    states = sorted(set(melody))
    indices = [states.index(pitch) for pitch in melody]
    counts = np.zeros((len(states),) * n)        # len(states) ** n entries!
    for i in range(len(indices) - n + 1):
        counts[tuple(indices[i:i + n])] += 1     # count every n-gram
    totals = counts.sum(axis=-1, keepdims=True)  # how often each context is followed by anything
    return states, counts / np.maximum(totals, 1)


def ngram(N: int, states: list[int], probabilities: np.ndarray,
          initial_state: tuple[int, ...] | None = None, ioi: float = 0.25,
          duration: float = 0.5) -> pq.Score:
    """A melody of up to N notes from an n-gram model, which starts with a context
    of n - 1 pitches (by default, one that occurs in the data, chosen at random)."""
    n = probabilities.ndim
    if initial_state is None:
        contexts = np.argwhere(probabilities.sum(axis=-1) > 0)
        sequence = list(contexts[np.random.randint(len(contexts))])
    else:
        sequence = [states.index(pitch) for pitch in initial_state]
    while len(sequence) < N:
        p = probabilities[tuple(sequence[len(sequence) - (n - 1):])]
        if p.sum() == 0:                         # a dead end: this context only ends the piece
            break
        sequence.append(np.random.choice(len(states), p=p))
    return pq.Score([(i * ioi, {"pitch": states[s], "duration": duration})
                     for i, s in enumerate(sequence)])


states, probabilities = compute_ngram("../assets/bach-prelude.mid", n=3)
score = ngram(64, states, probabilities, duration=1.0)
pq.play(score.render(pluck, metronome=pq.BasicMetronome(bpm=120)), normalize=True)

In [ ]:
states, probabilities = compute_ngram("../assets/mario.mid", n=3)
score = ngram(64, states, probabilities)
pq.play(score.render(pluck, metronome=pq.BasicMetronome(bpm=120)), normalize=True)